<a href="https://colab.research.google.com/github/EdwinRivera04/wildfire-smoke-detection/blob/main/notebooks/04_colab_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 04 — Colab Training (T4 GPU)
Run this notebook **top-to-bottom** on a Colab T4 runtime.  
Go to **Runtime → Change runtime type → T4 GPU** before starting.

In [1]:
# Cell 1 — Verify GPU
!nvidia-smi
import torch
assert torch.cuda.is_available(), "No GPU found — switch to T4 runtime (Runtime > Change runtime type)"
print(f"CUDA: {torch.version.cuda} | Device: {torch.cuda.get_device_name(0)}")

Tue Apr 21 04:42:29 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   34C    P0             56W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
# Cell 2 — Install dependencies
!pip install -q ultralytics albumentations scikit-learn

In [3]:
import os
REPO_URL = "https://github.com/EdwinRivera04/wildfire-smoke-detection.git"
REPO_DIR = "/content/wildfire-smoke-detection"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

%cd {REPO_DIR}
print("Working dir:", os.getcwd())

Cloning into '/content/wildfire-smoke-detection'...
remote: Enumerating objects: 66, done.
remote: Counting objects: 100% (66/66), done.
remote: Compressing objects: 100% (44/44), done.
remote: Total 66 (delta 18), reused 64 (delta 16), pack-reused 0 (from 0)
Receiving objects: 100% (66/66), 87.50 KiB | 410.00 KiB/s, done.
Resolving deltas: 100% (18/18), done.
/content/wildfire-smoke-detection
Working dir: /content/wildfire-smoke-detection


In [4]:
from google.colab import userdata
import os
os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"]      = userdata.get("KAGGLE_KEY")

!kaggle datasets download -d sayedgamal99/smoke-fire-detection-yolo -p data/
!unzip -q data/smoke-fire-detection-yolo.zip -d data/raw/
!rm data/smoke-fire-detection-yolo.zip
print("Dataset downloaded.")

Dataset URL: https://www.kaggle.com/datasets/sayedgamal99/smoke-fire-detection-yolo
License(s): CC0-1.0
100% 2.84G/2.84G [02:56<00:00, 17.3MB/s]

Dataset downloaded.


In [5]:
# Cell 5 — Run prepare_dataset.py to build data/processed/
!python src/data/prepare_dataset.py

train: 14122 images | 14122 valid pairs | 0 missing labels
val: 3099 images | 3099 valid pairs | 0 missing labels
test: 4306 images | 4306 valid pairs | 0 missing labels

Split      Total   Fire   Smoke   Both   Neither
--------------------------------------------------
train      14122   3836     770   3058      6458
val         3099    845     174    705      1375
test        4306   1186     220    895      2005

Dataset written to data/processed
Class mapping applied: raw 0=smoke→1, raw 1=fire→0 (project: 0=fire, 1=smoke)


In [14]:
# Cell 6 — Patch configs for CUDA (T4)
import yaml, copy

CUDA_OVERRIDES = {"device": "cuda", "workers": 4, "imgsz": 640, "batch": 16}
MPS_RESTORE    = {"device": "mps",  "workers": 0, "imgsz": 416, "batch": 32}

def patch_config(path, overrides):
    with open(path) as f:
        cfg = yaml.safe_load(f)
    cfg.update(overrides)
    with open(path, "w") as f:
        yaml.dump(cfg, f, default_flow_style=False)

patch_config("configs/baseline.yaml", CUDA_OVERRIDES)
patch_config("configs/improved.yaml", CUDA_OVERRIDES)
print("Configs patched for CUDA:")
!cat configs/baseline.yaml | grep -E 'device|workers|imgsz|batch'

Configs patched for CUDA:
batch: 16
device: cuda
imgsz: 640
workers: 4


In [7]:
# Cell 7 — Train baseline (25 epochs, ~17 min on T4)
import time
t0 = time.time()
!python src/train.py --config configs/baseline.yaml
baseline_time = time.time() - t0
print(f"\nBaseline training: {baseline_time/60:.1f} min")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.40 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data/processed/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=25, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v

In [15]:
# Cell 8 — Train improved (15 epochs, ~11 min on T4)
t0 = time.time()
!python src/train.py --config configs/improved.yaml
improved_time = time.time() - t0
print(f"\nImproved training: {improved_time/60:.1f} min")
print(f"Total: {(baseline_time + improved_time)/60:.1f} min")

augment.py not yet available — proceeding without custom augmentation.
Ultralytics 8.4.40 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=data/processed/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=15, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=improved, nbs=64, nms=Fa

In [18]:
# Cell 9 — Mount Google Drive and save weights
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

import shutil, glob
from pathlib import Path

drive_dir = Path("/content/drive/MyDrive/wildfire-weights")
drive_dir.mkdir(parents=True, exist_ok=True)

def find_best_pt(name):
    matches = glob.glob(f"**/{name}/weights/best.pt", recursive=True)
    if matches:
        return Path(sorted(matches)[-1])
    raise FileNotFoundError(f"No best.pt for '{name}'")

baseline_pt = find_best_pt("baseline")
improved_pt = find_best_pt("improved")

shutil.copy(baseline_pt, drive_dir / "baseline_best.pt")
shutil.copy(improved_pt, drive_dir / "improved_best.pt")
print(f"baseline : {baseline_pt}")
print(f"improved : {improved_pt}")
print(f"\nWeights saved to: {drive_dir}")
print("Share this Drive folder link with your teammates.")

Mounted at /content/drive
baseline : runs/detect/outputs/checkpoints/baseline/weights/best.pt
improved : runs/detect/outputs/checkpoints/improved/weights/best.pt

Weights saved to: /content/drive/MyDrive/wildfire-weights
Share this Drive folder link with your teammates.


In [23]:
# Cell 10 — Restore configs to MPS values and push
patch_config("configs/baseline.yaml", MPS_RESTORE)
patch_config("configs/improved.yaml", {**MPS_RESTORE, "batch": 32})
print("Configs restored to MPS values.")

# Commit restored configs
!git add configs/
!git commit -m "restore configs to MPS values after Colab training"
!git push

Configs restored to MPS values.
On branch main
Your branch is up to date with 'origin/main'.

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	outputs/
	runs/

nothing added to commit but untracked files present (use "git add" to track)
Everything up-to-date


In [22]:
# Cell 11 — Summary
print("=" * 60)
print("TRAINING COMPLETE")
print(f"  Baseline : {baseline_time/60:.1f} min")
print(f"  Improved : {improved_time/60:.1f} min")
print(f"  Total    : {(baseline_time+improved_time)/60:.1f} min")
print()
print(f"Weights saved to Google Drive: {drive_dir}")
# print()
# print("Next steps for teammates:")
# print("  1. Lyla shares the Drive folder link")
# print("  2. Edwin & Ann download baseline_best.pt and improved_best.pt")
# print("  3. Place at outputs/checkpoints/{baseline,improved}/weights/best.pt")
# print("  4. Ann runs: python src/evaluate.py")
# print("  5. Ann runs: python src/demo.py")
# print("=" * 60)

TRAINING COMPLETE
  Baseline : 49.0 min
  Improved : 34.6 min
  Total    : 83.6 min

Weights saved to Google Drive: /content/drive/MyDrive/wildfire-weights
